# Qwen2.5-VL OCR API Server

Deploy Qwen2.5-VL model as an OCR API on Google Colab.
The local `pdf-extractor.py` sends page images to this endpoint.

**Static ngrok URL:** `https://unhunted-olfactorily-roxanna.ngrok-free.dev/ocr`

**Usage:**
1. Get free ngrok auth token at https://dashboard.ngrok.com/get-started/your-authtoken
2. Paste token in Cell 2
3. Run all cells in order
4. Set `DEEPSEEK_OCR_URL=https://unhunted-olfactorily-roxanna.ngrok-free.dev/ocr` in your `.env` file

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers torch flask pyngrok Pillow qwen-vl-utils accelerate

In [ ]:
# Cell 2: Configure ngrok auth token (required for tunnel)
# Get your free token at: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = ""  # <-- Paste your token here

import os
os.environ["NGROK_AUTH_TOKEN"] = NGROK_AUTH_TOKEN

In [ ]:
# Cell 3: Load Qwen2.5-VL model for OCR
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"

print(f"Loading {MODEL_NAME}...")
processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()
print("Model loaded successfully!")

In [ ]:
# Cell 4: Define OCR inference function
import base64
import io
from PIL import Image
from qwen_vl_utils import process_vision_info


def run_ocr(image_b64: str) -> str:
    """Run OCR on a base64-encoded image using Qwen2.5-VL."""
    image_bytes = base64.b64decode(image_b64)
    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

    # Build chat messages with image + OCR prompt
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {
                    "type": "text",
                    "text": "Extract all text from this image. Return the text exactly as it appears, preserving the original language (Vietnamese).",
                },
            ],
        }
    ]

    # Prepare inputs using Qwen2.5-VL processor
    text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text_prompt],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2048,
            do_sample=False,
        )

    # Decode only the new tokens (skip the prompt)
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    text = processor.decode(generated, skip_special_tokens=True)
    return text.strip()

In [ ]:
# Cell 5: Flask API server + ngrok tunnel (static domain)
import time
import threading
from flask import Flask, request, jsonify
from pyngrok import ngrok

NGROK_DOMAIN = "unhunted-olfactorily-roxanna.ngrok-free.dev"

# Set ngrok auth token
ngrok.set_auth_token(os.environ["NGROK_AUTH_TOKEN"])

# Request tracking
stats = {"total": 0, "success": 0, "error": 0, "active": 0}
stats_lock = threading.Lock()

app = Flask(__name__)


@app.route("/ocr", methods=["POST"])
def ocr_endpoint():
    """Receive base64 image, return extracted text."""
    data = request.json
    if not data or "image" not in data:
        return jsonify({"error": "Missing 'image' field"}), 400

    with stats_lock:
        stats["total"] += 1
        stats["active"] += 1
        req_num = stats["total"]

    start = time.time()
    print(f"[REQ #{req_num}] Started | Active: {stats['active']}")

    try:
        text = run_ocr(data["image"])
        elapsed = time.time() - start
        with stats_lock:
            stats["success"] += 1
            stats["active"] -= 1
        print(f"[REQ #{req_num}] Done in {elapsed:.1f}s | {len(text)} chars | Active: {stats['active']}")
        return jsonify({"text": text})
    except Exception as exc:
        elapsed = time.time() - start
        with stats_lock:
            stats["error"] += 1
            stats["active"] -= 1
        print(f"[REQ #{req_num}] ERROR in {elapsed:.1f}s: {exc} | Active: {stats['active']}")
        return jsonify({"error": str(exc)}), 500


@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "model": MODEL_NAME, "stats": stats})


# Start ngrok tunnel with static domain
public_url = ngrok.connect(5000, domain=NGROK_DOMAIN)
print(f"\n{'='*60}")
print(f"OCR API URL: https://{NGROK_DOMAIN}/ocr")
print(f"Health check: https://{NGROK_DOMAIN}/health")
print(f"{'='*60}")

# Run Flask in a thread so the cell doesn't block
threading.Thread(
    target=lambda: app.run(port=5000, use_reloader=False),
    daemon=True,
).start()
print("\nServer running! Keep this notebook open.")